In [2]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.4 MB/s eta 0:00:00


In [3]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root = '/tmp/Cora', name = 'Cora')

Processing...
Done!


In [4]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

In [11]:
class GCN(torch.nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = GCNConv(dataset.num_node_features, 32)
    self.conv2 = GCNConv(32, 16)
    self.conv3 = GCNConv(16, dataset.num_classes)

  def forward(self, data):
    x, edge_index = data.x, data.edge_index

    x = self.conv1(x, edge_index)
    x = F.relu(x)
    x = F.dropout(x, training=self.training)
    x = self.conv2(x, edge_index)
    x = F.relu(x)
    x = F.dropout(x, training=self.training)
    x = self.conv3(x, edge_index)

    return F.log_softmax(x, dim = 1)

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GCN().to(device)
data = dataset[0].to(device)
opt = torch.optim.Adam(model.parameters(), lr = 0.01, weight_decay=5e-4)

In [19]:
model.train()
for e in range(100):
  opt.zero_grad()
  out = model(data)
  loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
  loss.backward()
  opt.step()

`dim = -1 doesn't mean just the "3rd column"—it means across all columns for every row.`

In [20]:
model.eval()
pred = model(data).argmax(dim=1)
correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
acc = int(correct) / int(data.test_mask.sum())

In [21]:
print(f'Accuracy: {acc:.4f}')

Accuracy: 0.7860
